# N11 · GPU 内存层级与 Roofline

**关联 lab**: L01.7 GPU 内存层级与最小 Triton kernel

**学习目标**: 在跑 Triton kernel 之前，先用纸笔/numpy 把性能上界算出来。理解 fused softmax 的提速来自哪一层存储，以及 BLOCK_SIZE / num_warps 如何决定 occupancy。

**No-GPU 可完成度**: 100%。本 notebook 全部是数学与配置，CPU 即可跑。

**对应 MiniInfra 模块**:
- `mini_infra/gpu/memory_model.py`: `GPU_PROFILES`, `softmax_bytes`, `occupancy_estimate`, `roofline_softmax`

**对应真实源码**:
- `github_repo/triton/python/tutorials/02-fused-softmax.py`
- `github_repo/Megatron-LM/megatron/core/fusions/fused_softmax.py`


## 1. GPU 内存层级速览

现代 GPU 有四层存储，从慢到快：

| 层级 | 容量 (4090) | 带宽 (4090) | 作用 |
|---|---|---|---|
| HBM (global memory) | 24 GB | ~1008 GB/s | 模型权重、KV cache、激活 |
| L2 cache | 72 MB | ~5 TB/s | 跨 SM 共享，硬件管理 |
| SMEM (shared memory) | ~100 KB / SM | ~20 TB/s | block 内手动管理 |
| Registers | ~256 KB / SM | 接近无限 | warp/线程私有 |

**关键直觉**: 数量级差距是 **1 : 5 : 20 : 100+**。一个 kernel 如果能把同一份数据从 HBM 读上来后留在 SMEM 多次复用，速度就能逼近 SMEM 带宽而不是 HBM 带宽。这就是 "fused" 的本质。

先把 `GPU_PROFILES` 里的规格读出来对比：

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mini_infra.gpu.memory_model import GPU_PROFILES, softmax_bytes, occupancy_estimate, roofline_softmax

for name, profile in GPU_PROFILES.items():
    print(f'{name:18s}  HBM={profile.peak_hbm_gbs:>6.0f} GB/s  L2={profile.l2_mb:>4.0f} MB  '
          f'SMEM/SM={profile.smem_kb_per_sm:>5.0f} KB  warps/SM={profile.max_warps_per_sm}')

## 2. Roofline 与 Arithmetic Intensity

Roofline 模型说：一个 kernel 能达到的 GFLOPS 受两个上限约束：

$$\text{GFLOPS}_{\text{achievable}} = \min(\text{peak FLOPS}, \text{peak HBM bandwidth} \times \text{arithmetic intensity})$$

其中 arithmetic intensity = FLOP / Byte。

对 softmax 来说：
- 一行有 N 个元素，需要 ~5N 次浮点（exp、减、除、加、reduce）
- fused softmax 的 HBM 字节 ≈ 2N × dtype_bytes（一次读 + 一次写）
- 所以 AI = 5N / (2N × 2) = **1.25 FLOP/Byte**（fp16）

对照：matmul 的 AI 通常是 hidden 量级（数百到数千）——所以 matmul 是 compute-bound，softmax 是 memory-bound。

**结论**: softmax 的优化方向不是堆算力，而是降 HBM 字节。这正是 fused 的胜利点。

## 3. 字节预算：fused vs eager

PyTorch eager softmax 大致是这样：
```
x_max = x.max(dim=-1, keepdim=True)   # 读 X，写 max
e = (x - x_max).exp()                 # 读 X 与 max，写 e
out = e / e.sum(dim=-1, keepdim=True) # 读 e，写 sum，再读 e + 写 out
```
中间张量 `x_max`、`e`、`sum` 都得在 HBM 来回。fused 把所有中间量留在 SMEM/寄存器里，HBM 只读 X、写 out。

用 `softmax_bytes` 算具体差：

In [ ]:
for batch, seq_len in [(8, 1024), (8, 4096), (8, 16384)]:
    fused = softmax_bytes(batch, seq_len, dtype_bytes=2, fused=True)
    eager = softmax_bytes(batch, seq_len, dtype_bytes=2, fused=False)
    ratio = eager / fused
    print(f'batch={batch} seq={seq_len:>5}  fused={fused/1e6:>6.2f} MB  eager={eager/1e6:>6.2f} MB  '
          f'ratio=eager/{ratio:.2f}x')

**关键观察**: eager 大约多读 2.5x 字节。在 memory-bound 区，这直接对应 ~2.5x 的延迟差。

把 fused 字节数除以 4090 peak HBM (1008 GB/s) 就得到理论最快延迟：

In [ ]:
for seq_len in [1024, 4096, 16384]:
    bytes_fused = softmax_bytes(8, seq_len, dtype_bytes=2, fused=True)
    peak_ms = bytes_fused / (1008 * 1e9) * 1000
    print(f'seq={seq_len:>5}  fused={bytes_fused/1e6:>6.2f} MB  '
          f'fastest possible at 4090 = {peak_ms:.4f} ms')

## 4. Occupancy 与 BLOCK_SIZE

Occupancy 是 SM 上活跃 warp 数 / 上限的比值。它受三个约束：

1. **SMEM**: BLOCK_SIZE 太大 → 一个 block 占的 SMEM 太多 → 同一 SM 上能驻留的 block 数变少
2. **Registers**: num_warps × BLOCK_SIZE 决定每个 block 的寄存器占用
3. **Resident block 上限**: 硬件限制，通常 8/SM

用 `occupancy_estimate` 看 4090 上不同 BLOCK_SIZE 的估算：

In [ ]:
profile_4090 = GPU_PROFILES['rtx4090']
print(f'{"BLOCK_SIZE":>10}  {"num_warps":>10}  {"occupancy":>10}')
for block_size in [256, 512, 1024, 2048, 4096]:
    for num_warps in [2, 4, 8]:
        occ = occupancy_estimate(block_size, num_warps, profile_4090)
        print(f'{block_size:>10}  {num_warps:>10}  {occ:>10.3f}')

**直觉**: occupancy 不是越高越好，但 < 0.5 通常意味着 SM 利用率受限。BLOCK_SIZE=1024 + num_warps=4 在 4090 上是稳妥起点。

**注意**: `occupancy_estimate` 是教学启发式，不是 ncu 的 achieved occupancy。真实运行时请用 `ncu --set basic` 覆盖。

## 5. Roofline 综合预算

把字节、occupancy、achieved bandwidth 比例一起算：

In [ ]:
import json

for device in ['rtx4090', 'h200']:
    print(f'\n=== {device} ===')
    for block_size in [512, 1024, 2048]:
        plan = roofline_softmax(batch=8, seq_len=4096, block_size=block_size, device=device, num_warps=4)
        print(f'BLOCK={block_size:>4}  bw={plan["bandwidth_gbs"]:>6.0f} GB/s '
              f'({plan["bandwidth_gbs"]/plan["peak_hbm_gbs"]*100:>4.1f}% peak)  '
              f'occ={plan["occupancy"]:.2f}  '
              f'speedup_vs_torch={plan["speedup_vs_torch"]}x')

## 6. 自检问题（运行后回答）

1. 对 `batch=8, seq_len=16384, fp16` 的 fused softmax，4090 上理论最快延迟是多少 ms？eager 大约多多少倍？
2. BLOCK_SIZE 从 1024 升到 2048，occupancy 通常会怎么变？为什么？
3. 为什么 softmax 是 memory-bound 而不是 compute-bound？给出 arithmetic intensity 的具体数字。
4. H200 的 peak HBM 是 4090 的几倍？理论上 fused softmax 在 H200 上应该比 4090 快几倍？（提示：4800/1008 ≈ 4.76x）
5. `occupancy_estimate` 给出 0.6 时，你下一步是调 BLOCK_SIZE 还是 num_warps？

## 7. 与 lab smoke 的对接

下一步运行 lab smoke 时，你应该带着这些预测：

```bash
make mini-infra M=l04_gpu_kernel RUN_ID=demo
torchrun --nproc_per_node=1 labs/l04_gpu_kernel/scripts/bench_softmax.py \
  --seq 4096 --block 1024 --num-warps 4
```

把实测 `bandwidth_gbs` 与本 notebook 的 `roofline_softmax` 估算对照。常见偏差来源：
- L2 cache 命中让实测 > peak HBM 估算（这是好的）
- launch overhead 让小 batch 实测远低于估算（用更大 N 验证）
- BLOCK_SIZE 没在甜区 → 与 ticket `kernel_low_bw_003` 对照修复

在 `report.md` 中写清估算与实测的偏差，并解释原因。